# Graph Convolutional Networks (GCN) for Traffic Flow Prediction
## A Complete Learning Guide

**Objective:** Understand and implement a Graph Convolutional Network for predicting traffic speeds on the METR-LA dataset.

### What You'll Learn:
1. **Graph Neural Networks Fundamentals** - How graphs differ from regular data
2. **GCN Architecture** - The mathematics and implementation
3. **Spatial-Temporal Modeling** - Combining graph structure with temporal features
4. **Training & Evaluation** - Building a complete ML pipeline
5. **Inference** - Making predictions on new data

## Part 1: Core Concepts & Setup

### 1.1 What is a Graph?

A **graph** is a data structure consisting of:
- **Nodes** (vertices): Traffic sensors at different highway locations
- **Edges** (connections): Physical proximity or road connections between sensors
- **Edge Weights**: Strength of connections (e.g., inverse distance)

Unlike images (grid structure) or sequences (linear), graphs capture **irregular spatial relationships**.

### Why Graphs for Traffic?
- Traffic doesn't flow in grids—it follows actual road networks
- Adjacent highways influence each other's traffic
- A GCN learns from neighboring sensors to predict future speeds

### 1.2 Install Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

packages = ['numpy==1.26.4', 'scipy==1.12.0', 'torch==2.2.2', 'scikit-learn', 'pandas']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print("✓ All dependencies installed!")

### 1.3 Import Libraries

In [ ]:
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module
import scipy.sparse as sp
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 2: Building the GCN Layer from Scratch

### 2.1 Understanding Graph Convolution

A **Graph Convolution** operation works like this:

```
h_i^(l+1) = σ(W^l * h_i^(l) + Σ_{j∈N(i)} W^l * h_j^(l))
```

Where:
- `h_i^(l)`: hidden state of node i at layer l
- `N(i)`: neighbors of node i
- `W`: learnable weight matrix
- `σ`: activation function (ReLU)

**In matrix form:**
```
H^(l+1) = D^(-1/2) A D^(-1/2) H^(l) W^l
```

Where:
- `A`: Adjacency matrix (graph connections)
- `D`: Degree matrix (normalization)
- `H`: Feature matrix (node attributes)
- `W`: Learnable weights

In [ ]:
class GraphConvolution(Module):
    """
    Simple GCN layer, similar to https://arxiv.org/abs/1609.02907
    
    This layer performs one step of message passing on a graph:
    1. Each node transforms its features with a learned weight matrix
    2. Aggregates information from neighboring nodes via the adjacency matrix
    """

    def __init__(self, in_features, out_features, bias=True):
        """
        Initialize a Graph Convolution Layer
        
        Args:
            in_features (int): Size of input features per node
            out_features (int): Size of output features per node
            bias (bool): Whether to add a bias term
        """
        super(GraphConvolution, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Learnable weight matrix: (in_features, out_features)
        self.weight = Parameter(torch.FloatTensor(in_features, out_features))
        
        if bias:
            self.bias = Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        
        # Initialize parameters with small random values
        self.reset_parameters()

    def reset_parameters(self):
        """
        Initialize weights using uniform distribution.
        Prevents dead neurons and ensures stable training.
        """
        stdv = 1.0 / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

    def forward(self, input, adj):
        """
        Forward pass of Graph Convolution
        
        Args:
            input (Tensor): Node features, shape (num_nodes, in_features)
            adj (SparseTensor): Adjacency matrix, shape (num_nodes, num_nodes)
        
        Returns:
            output (Tensor): Aggregated features, shape (num_nodes, out_features)
        """
        # Step 1: Transform node features
        # input @ weight: (num_nodes, in_features) @ (in_features, out_features)
        #              -> (num_nodes, out_features)
        support = torch.mm(input, self.weight)
        
        # Step 2: Aggregate from neighbors
        # adj @ support: Apply adjacency matrix to propagate information
        # (num_nodes, num_nodes) @ (num_nodes, out_features)
        #              -> (num_nodes, out_features)
        output = torch.spmm(adj, support)  # spmm = sparse matrix multiplication
        
        # Step 3: Add bias if applicable
        if self.bias is not None:
            output = output + self.bias
        
        return output

    def __repr__(self):
        return self.__class__.__name__ + ' (' + str(self.in_features) + ' -> ' + str(self.out_features) + ')'


# Test the layer
print("✓ GraphConvolution layer defined")

# Example: Create a GCN layer that transforms 10 features to 8 features
example_layer = GraphConvolution(in_features=10, out_features=8)
print(f"\nExample layer: {example_layer}")
print(f"Weight shape: {example_layer.weight.shape}")
print(f"Bias shape: {example_layer.bias.shape}")

### 2.2 Understanding the Full GCN Model

In [ ]:
class GCN(nn.Module):
    """
    Two-layer Graph Convolutional Network with Residual Connection
    
    Architecture:
    ┌─────────────────────────────┐
    │ Input Features (12 nodes)   │
    └──────────────┬──────────────┘
                   │
         ┌─────────┴─────────┐
         │                   │
    ┌────▼────┐         ┌────▼────┐
    │  GCN1   │         │ Residual│  (Path B: Direct)
    │(12→32)  │         │(12→1)   │
    └────┬────┘         └────┬────┘
         │                   │
         │ ReLU + Dropout    │
    ┌────▼────┐              │
    │  GCN2   │              │
    │(32→1)   │              │
    └────┬────┴──────────────┘
         │     +              
    ┌────▼────────────────┐
    │  Output (1 value)   │ ← Speed prediction
    └─────────────────────┘
    """

    def __init__(self, nfeat, nhid1, nclass, dropout):
        """
        Args:
            nfeat (int): Number of input features per node (temporal window)
            nhid1 (int): Number of hidden units in first GCN layer
            nclass (int): Number of output classes/values (1 for regression)
            dropout (float): Dropout rate (0.0 to 1.0)
        """
        super(GCN, self).__init__()
        
        # Path A: Spatial message aggregation through graph
        self.gc1 = GraphConvolution(nfeat, nhid1)      # 12 → 32
        self.gc2 = GraphConvolution(nhid1, nclass)     # 32 → 1
        
        # Path B: Direct feature transformation (residual/skip connection)
        # Allows original information to flow directly to output
        self.residual = nn.Linear(nfeat, nclass)       # 12 → 1
        
        self.dropout = dropout
        
        # Initialize residual weights carefully
        nn.init.xavier_uniform_(self.residual.weight)

    def forward(self, x, adj):
        """
        Forward pass
        
        Args:
            x (Tensor): Node features, shape (num_nodes, nfeat)
            adj (SparseTensor): Adjacency matrix, shape (num_nodes, num_nodes)
        
        Returns:
            output (Tensor): Predicted speeds, shape (num_nodes, 1)
        """
        # Path A: Graph convolution with non-linearity
        gcn_path = self.gc1(x, adj)              # (num_nodes, nfeat) → (num_nodes, nhid1)
        gcn_path = F.relu(gcn_path)              # Apply ReLU activation
        gcn_path = F.dropout(gcn_path, p=self.dropout, training=self.training)  # Regularization
        gcn_path = self.gc2(gcn_path, adj)       # (num_nodes, nhid1) → (num_nodes, nclass)
        
        # Path B: Direct feature transformation
        skip_path = self.residual(x)             # (num_nodes, nfeat) → (num_nodes, nclass)
        
        # Combine both paths
        output = gcn_path + skip_path
        
        return output


# Test the model
print("✓ GCN model defined")

# Create a small example
model = GCN(nfeat=12, nhid1=32, nclass=1, dropout=0.3)
print(f"\nModel architecture:")
print(model)

## Part 3: Data Preparation & Loading

### 3.1 Understanding the Dataset

**METR-LA Dataset Components:**

1. **graph_sensor_locations.csv** - Maps sensor IDs to geographic coordinates
   - 207 sensors monitoring traffic across LA
   - Each row: index, sensor_id, latitude, longitude

2. **distances_la_2012.csv** - Physical distances between sensors
   - Captures road network topology
   - Each row: from_sensor, to_sensor, distance

3. **W_metrla.csv** - Pre-computed normalized weight matrix
   - Graph connections weighted by proximity
   - 207×207 sparse matrix

4. **Temporal Features** - Historical speed measurements
   - 12 time steps of historical speeds
   - Predict next speed value

In [ ]:
def get_absolute_path(filename):
    """
    Get absolute path to data files.
    NOTE: Modify this path to match your local data directory!
    """
    # For Jupyter notebooks, typically in the same directory
    import os
    return os.path.join(os.getcwd(), filename)

def clean_id(raw_val):
    """
    Safely clean and standardize IDs for consistent matching.
    Handles strings like '123', '123.0', and float 123.0 uniformly.
    """
    try:
        return str(int(float(raw_val)))
    except:
        return str(raw_val).strip()

print("✓ Helper functions defined")

### 3.2 Build the Graph from Distance Data

In [ ]:
def symmetric_normalize(mx):
    """
    Normalize adjacency matrix: D^(-1/2) A D^(-1/2)
    
    This is critical for GCN:
    - Prevents gradient explosion/vanishing
    - Scales connections by node degree
    - Makes graph convolution well-behaved
    
    Args:
        mx (scipy.sparse): Sparse adjacency matrix
    
    Returns:
        Normalized sparse matrix
    """
    # Calculate degree (row sum)
    rowsum = np.array(mx.sum(1))
    
    # Compute D^(-1/2)
    r_inv_sqrt = np.power(rowsum, -0.5).flatten()
    
    # Guard against infinity and NaN
    r_inv_sqrt[np.isinf(r_inv_sqrt) | np.isnan(r_inv_sqrt)] = 0.
    
    # Create diagonal matrix
    r_mat_inv_sqrt = sp.diags(r_inv_sqrt)
    
    # Apply normalization: D^(-1/2) A D^(-1/2)
    return r_mat_inv_sqrt.dot(mx).dot(r_mat_inv_sqrt)

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    """
    Convert SciPy sparse matrix to PyTorch sparse tensor.
    
    PyTorch sparse tensors enable efficient graph convolution
    on large sparse graphs.
    
    Args:
        sparse_mx: SciPy sparse matrix
    
    Returns:
        PyTorch sparse COO tensor
    """
    # Convert to COO format (easier to access indices and values)
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    
    # Extract indices: [[row1, row2, ...], [col1, col2, ...]]
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    
    # Extract values
    values = torch.from_numpy(sparse_mx.data)
    
    # Create sparse tensor
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse_coo_tensor(indices, values, shape, dtype=torch.float32)

print("✓ Graph normalization utilities defined")

### 3.3 Load and Prepare Data

In [ ]:
def load_data(data_dir="."):
    """
    Load and construct graph structure for METR-LA traffic prediction.
    
    Steps:
    1. Parse sensor locations
    2. Build distance matrix from topology file
    3. Construct K-NN adjacency with inverse distance weights
    4. Normalize the graph
    5. Generate synthetic temporal features for demo
    
    Returns:
        adj (torch.sparse): Normalized adjacency matrix
        features (torch.Tensor): Node features (num_nodes, num_features)
        labels (torch.Tensor): Target speeds (num_nodes, 1)
        feature_cols (list): Feature names
    """
    print("Loading spatial graph using CSV locations and K-Nearest Neighbors...")
    
    dist_file = get_absolute_path("distances_la_2012.csv")
    ids_file = get_absolute_path("graph_sensor_locations.csv")
    
    # ===== STEP 1: Load Sensor IDs =====
    print("\n[STEP 1] Loading sensor locations...")
    try:
        loc_df = pd.read_csv(ids_file)
        sensor_ids = [clean_id(x) for x in loc_df.iloc[:, 0].values]
    except FileNotFoundError:
        print("⚠ CSV files not found. Using synthetic data for demonstration...")
        # Generate synthetic data for demo
        num_nodes = 207
        sensor_ids = [str(i) for i in range(num_nodes)]
    
    sensor_id_to_ind = {str(sid): i for i, sid in enumerate(sensor_ids)}
    num_nodes = len(sensor_ids)
    print(f"✓ Extracted {num_nodes} valid sensor IDs")
    
    # ===== STEP 2: Build Distance Matrix =====
    print("\n[STEP 2] Building distance matrix...")
    dist_mx = np.full((num_nodes, num_nodes), np.inf, dtype=np.float32)
    np.fill_diagonal(dist_mx, 0.0)
    
    try:
        dist_df = pd.read_csv(dist_file)
        for _, row in dist_df.iterrows():
            src_id, dst_id = clean_id(row.iloc[0]), clean_id(row.iloc[1])
            cost = float(row.iloc[2])
            if src_id in sensor_id_to_ind and dst_id in sensor_id_to_ind:
                dist_mx[sensor_id_to_ind[src_id], sensor_id_to_ind[dst_id]] = cost
        print(f"✓ Distance matrix constructed")
    except FileNotFoundError:
        print("⚠ Distance file not found. Using random distances...")
        np.random.seed(42)
        for i in range(num_nodes):
            for j in range(num_nodes):
                if i != j:
                    dist_mx[i, j] = np.random.uniform(1, 20)
    
    # ===== STEP 3: Construct K-NN Graph =====
    print("\n[STEP 3] Constructing K-Nearest Neighbors graph...")
    adj_dense = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    k_neighbors = 15
    
    for i in range(num_nodes):
        # Find k nearest neighbors
        node_distances = dist_mx[i, :]
        closest_indices = np.argsort(node_distances)
        
        # Connect to k nearest neighbors
        connected_count = 0
        for j in closest_indices:
            if i != j and node_distances[j] != np.inf:
                # Weight inversely to distance: closer nodes = stronger connection
                adj_dense[i, j] = 1.0 / (node_distances[j] + 1.0)
                connected_count += 1
            
            if connected_count >= k_neighbors:
                break
    
    print(f"✓ K-NN graph with k={k_neighbors} neighbors constructed")
    
    # ===== STEP 4: Normalize Adjacency Matrix =====
    print("\n[STEP 4] Normalizing adjacency matrix...")
    adj = sp.coo_matrix(adj_dense, dtype=np.float32)
    adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)  # Symmetrize
    adj = adj + sp.eye(num_nodes)  # Add self-loops
    adj = symmetric_normalize(adj)
    print(f"✓ Normalized adjacency matrix")
    print(f"  - Shape: {adj.shape}")
    print(f"  - Sparsity: {1 - (adj.nnz / (adj.shape[0] * adj.shape[1]))*100:.2f}%")
    
    # ===== STEP 5: Generate Features & Labels =====
    print("\n[STEP 5] Generating temporal features...")
    np.random.seed(42)
    num_features = 12  # 12 time steps of history
    
    features_np = np.zeros((num_nodes, num_features), dtype=np.float32)
    labels_np = np.zeros((num_nodes, 1), dtype=np.float32)
    
    # Synthetic speed data following a sinusoidal pattern
    for node in range(num_nodes):
        for t in range(num_features):
            # Simulate traffic speed with temporal and spatial variation
            features_np[node, t] = 55.0 + 12.0 * np.sin(t * 0.4 - node * 0.2) + np.random.normal(0, 0.1)
        
        # Target: predicted next speed
        labels_np[node, 0] = 55.0 + 12.0 * np.sin(num_features * 0.4 - node * 0.2)
    
    print(f"✓ Features generated: shape {features_np.shape}")
    print(f"  - Speed range: [{features_np.min():.1f}, {features_np.max():.1f}] mph")
    
    # Convert to PyTorch tensors
    features = torch.FloatTensor(features_np)
    labels = torch.FloatTensor(labels_np)
    adj = sparse_mx_to_torch_sparse_tensor(adj)
    
    feature_cols = [f"speed_t_{i}" for i in range(num_features)]
    
    print(f"\n✓ Data loading complete!")
    return adj, features, labels, feature_cols


# Load data
print("="*60)
adj, features, labels, feature_cols = load_data()
print("="*60)

## Part 4: Training the GCN Model

### 4.1 Training Configuration

In [ ]:
# Configuration
config = {
    'seed': 42,
    'epochs': 150,
    'learning_rate': 0.001,
    'weight_decay': 1e-4,
    'hidden_units': 32,
    'dropout': 0.3,
    'use_cuda': torch.cuda.is_available(),
    'k_folds': 5,
}

# Set random seed for reproducibility
torch.manual_seed(config['seed'])
np.random.seed(config['seed'])
if config['use_cuda']:
    torch.cuda.manual_seed(config['seed'])

print("Training Configuration:")
print("-" * 40)
for key, value in config.items():
    print(f"{key:.<30} {value}")

### 4.2 Define Training Functions

In [ ]:
def train_epoch(model, optimizer, features, adj, labels, idx_train):
    """
    Train for one epoch on training subset.
    
    Args:
        model: GCN model
        optimizer: Adam optimizer
        features: Node features tensor
        adj: Adjacency matrix
        labels: Target labels
        idx_train: Indices of training nodes
    
    Returns:
        Training loss
    """
    model.train()
    optimizer.zero_grad()
    
    # Forward pass: predict for all nodes
    output = model(features, adj)
    
    # Compute loss on training subset only
    loss_train = F.mse_loss(output[idx_train], labels[idx_train])
    
    # Backward pass
    loss_train.backward()
    
    # Gradient clipping to prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # Update weights
    optimizer.step()
    
    return loss_train.item()


def validate_epoch(model, features, adj, labels, idx_val):
    """
    Evaluate on validation subset.
    
    Metrics:
    - MSE Loss: Mean squared error
    - MAE: Mean absolute error
    - R² (Accuracy): Coefficient of determination
    """
    model.eval()
    with torch.no_grad():
        output = model(features, adj)
        
        # MSE Loss
        loss_val = F.mse_loss(output[idx_val], labels[idx_val])
        
        # MAE
        mae_val = torch.mean(torch.abs(output[idx_val] - labels[idx_val]))
        
        # R² Score: measures proportion of variance explained
        y_true = labels[idx_val]
        ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
        ss_res = torch.sum((y_true - output[idx_val]) ** 2)
        r2_score = 1 - (ss_res / (ss_tot + 1e-8))
    
    return loss_val.item(), mae_val.item(), r2_score.item()


def test_fold(model, features, adj, labels, idx_test):
    """
    Evaluate on test subset (final evaluation).
    """
    model.eval()
    with torch.no_grad():
        output = model(features, adj)
        
        loss_test = F.mse_loss(output[idx_test], labels[idx_test])
        mae_test = torch.mean(torch.abs(output[idx_test] - labels[idx_test]))
        
        y_true = labels[idx_test]
        ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
        ss_res = torch.sum((y_true - output[idx_test]) ** 2)
        r2_score = 1 - (ss_res / (ss_tot + 1e-8))
    
    return loss_test.item(), mae_test.item(), r2_score.item()


print("✓ Training functions defined")

### 4.3 Execute Training with K-Fold Cross-Validation

In [ ]:
# Move data to device
if config['use_cuda']:
    features = features.cuda()
    adj = adj.cuda()
    labels = labels.cuda()

# Display graph information
num_nodes = features.shape[0]
n_input_features = features.shape[1]

print("\n" + "="*60)
print("GRAPH ARCHITECTURE SUMMARY:")
print("="*60)
print(f"Total Nodes (Traffic Sensors):     {num_nodes}")
print(f"Input Features per Node:           {n_input_features} (temporal history)")
print(f"Hidden Units (GCN layer 1):        {config['hidden_units']}")
print(f"Output Dimensions:                 1 (speed prediction)")
print(f"Total Model Parameters:            {sum(p.numel() for p in nn.Linear(n_input_features, 1).parameters())} + graph layers")
print("="*60)

# K-Fold Cross-Validation
kf = KFold(n_splits=config['k_folds'], shuffle=True, random_state=config['seed'])
cv_accuracies = []
cv_maes = []
fold_histories = []

print(f"\nExecuting {config['k_folds']}-Fold Cross-Validation...\n")

for fold, (train_and_val_indices, test_indices) in enumerate(kf.split(np.arange(num_nodes))):
    print(f"{'='*60}")
    print(f"FOLD {fold + 1} / {config['k_folds']}")
    print(f"{'='*60}")
    
    # Split train+val into train and validation
    np.random.shuffle(train_and_val_indices)
    split_point = int(0.85 * len(train_and_val_indices))
    
    idx_train = torch.LongTensor(train_and_val_indices[:split_point])
    idx_val = torch.LongTensor(train_and_val_indices[split_point:])
    idx_test = torch.LongTensor(test_indices)
    
    if config['use_cuda']:
        idx_train = idx_train.cuda()
        idx_val = idx_val.cuda()
        idx_test = idx_test.cuda()
    
    print(f"Train nodes: {len(idx_train)} | Val nodes: {len(idx_val)} | Test nodes: {len(idx_test)}")
    
    # Initialize model
    model = GCN(nfeat=n_input_features, nhid1=config['hidden_units'], nclass=1, dropout=config['dropout'])
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    
    if config['use_cuda']:
        model.cuda()
    
    # Training loop
    history = {'loss': [], 'val_loss': [], 'val_r2': []}
    
    for epoch in range(config['epochs']):
        loss_t = train_epoch(model, optimizer, features, adj, labels, idx_train)
        loss_v, mae_v, acc_v = validate_epoch(model, features, adj, labels, idx_val)
        
        history['loss'].append(loss_t)
        history['val_loss'].append(loss_v)
        history['val_r2'].append(acc_v)
        
        # Print progress
        if (epoch + 1) % 30 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d} | Train Loss: {loss_t:.4f} | Val Loss: {loss_v:.4f} | Val R²: {acc_v:.4f}")
    
    # Evaluate on test set
    loss_te, mae_te, acc_te = test_fold(model, features, adj, labels, idx_test)
    print(f"\n✓ Test Results: MSE={loss_te:.4f} | MAE={mae_te:.4f} | R²={acc_te:.4f}")
    
    cv_maes.append(mae_te)
    cv_accuracies.append(acc_te)
    fold_histories.append(history)

# Final Summary
print(f"\n{'='*60}")
print(f"CROSS-VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Average Test MAE: {np.mean(cv_maes):.4f} (±{np.std(cv_maes):.4f})")
print(f"Average Test R²:  {np.mean(cv_accuracies):.4f} (±{np.std(cv_accuracies):.4f})")
print(f"{'='*60}")

# Save the final model
print("\nSaving final model weights...")
torch.save(model.state_dict(), "gcn_traffic_model.pth")
print("✓ Model saved as 'gcn_traffic_model.pth'")

## Part 5: Visualization & Analysis

### 5.1 Training History Visualization

In [ ]:
# Plot training history for first fold
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

history = fold_histories[0]

# Loss curves
axes[0].plot(history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Loss During Training (Fold 1)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# R² Score
axes[1].plot(history['val_r2'], label='Validation R²', linewidth=2, color='green')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_title('Model Performance (Fold 1)', fontsize=14, fontweight='bold')
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training curves plotted")

### 5.2 Cross-Validation Results

In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(10, 6))

folds = np.arange(1, config['k_folds'] + 1)
x = np.arange(len(folds))
width = 0.35

bars1 = ax.bar(x - width/2, cv_maes, width, label='MAE (mph)', alpha=0.8)
ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, cv_accuracies, width, label='R² Score', alpha=0.8, color='orange')

ax.set_xlabel('Fold', fontsize=12, fontweight='bold')
ax.set_ylabel('MAE (mph)', fontsize=11, fontweight='bold')
ax2.set_ylabel('R² Score', fontsize=11, fontweight='bold')
ax.set_title(f'{config["k_folds"]}-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(folds)

# Add legends
ax.legend(loc='upper left', fontsize=10)
ax2.legend(loc='upper right', fontsize=10)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("✓ Cross-validation results visualized")

## Part 6: Making Predictions (Inference)

### 6.1 Load Trained Model and Make Predictions

In [ ]:
def run_inference(model_path="gcn_traffic_model.pth"):
    """
    Load trained model and make predictions on current sensor data.
    """
    print("="*60)
    print("INFERENCE: Traffic Speed Forecasting")
    print("="*60)
    
    # Load data
    print("\n[1] Loading spatial graph and sensor data...")
    adj, features, _, _ = load_data()
    num_nodes = features.shape[0]
    n_input_features = features.shape[1]
    
    if config['use_cuda']:
        features = features.cuda()
        adj = adj.cuda()
    
    # Rebuild and load model
    print("\n[2] Loading trained model...")
    model = GCN(nfeat=n_input_features, nhid1=config['hidden_units'], nclass=1, dropout=0.0)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    
    if config['use_cuda']:
        model.cuda()
    
    # Set to evaluation mode
    model.eval()
    print("✓ Model loaded and ready for inference")
    
    # Run inference
    print("\n[3] Executing spatio-temporal forecast...")
    with torch.no_grad():
        predictions = model(features, adj)
    
    print("✓ Predictions generated for all sensors")
    
    # Display sample predictions
    print(f"\n{'-'*60}")
    print(f"LIVE PREDICTION RESULTS (Next Time Step)")
    print(f"{'-'*60}")
    print(f"{'Sensor':>8} {'Current Avg Speed (mph)':>25} {'Predicted Speed (mph)':>25}")
    print(f"{'-'*60}")
    
    for sensor_id in range(min(10, num_nodes)):
        current_avg = torch.mean(features[sensor_id]).item()
        predicted = predictions[sensor_id].item()
        print(f"  {sensor_id:>6} {current_avg:>24.2f} {predicted:>24.2f}")
    
    print(f"{'-'*60}")
    
    # Statistics
    print(f"\nPrediction Statistics:")
    print(f"  - Mean predicted speed: {predictions.mean():.2f} mph")
    print(f"  - Std predicted speed:  {predictions.std():.2f} mph")
    print(f"  - Min predicted speed:  {predictions.min():.2f} mph")
    print(f"  - Max predicted speed:  {predictions.max():.2f} mph")
    
    return predictions


# Run inference
predictions = run_inference()

### 6.2 Visualize Predictions

In [ ]:
# Convert predictions to numpy
if config['use_cuda']:
    pred_np = predictions.cpu().detach().numpy()
    true_np = labels.cpu().detach().numpy()
else:
    pred_np = predictions.detach().numpy()
    true_np = labels.detach().numpy()

# Create comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Prediction vs Actual
ax = axes[0, 0]
ax.scatter(true_np, pred_np, alpha=0.5, s=50)
min_val = min(true_np.min(), pred_np.min())
max_val = max(true_np.max(), pred_np.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Speed (mph)', fontsize=11, fontweight='bold')
ax.set_ylabel('Predicted Speed (mph)', fontsize=11, fontweight='bold')
ax.set_title('Actual vs Predicted Speeds', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Residuals
ax = axes[0, 1]
residuals = pred_np - true_np
ax.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='r', linestyle='--', lw=2)
ax.set_xlabel('Prediction Error (mph)', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('Distribution of Prediction Errors', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 3. Sample predictions across sensors
ax = axes[1, 0]
n_samples = 30
x = np.arange(n_samples)
ax.plot(x, true_np[:n_samples], 'o-', label='Actual', linewidth=2, markersize=6)
ax.plot(x, pred_np[:n_samples], 's--', label='Predicted', linewidth=2, markersize=6)
ax.set_xlabel('Sensor ID', fontsize=11, fontweight='bold')
ax.set_ylabel('Speed (mph)', fontsize=11, fontweight='bold')
ax.set_title(f'Sample Predictions (First {n_samples} Sensors)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Error vs Actual Speed
ax = axes[1, 1]
ax.scatter(true_np, np.abs(residuals), alpha=0.5, s=50)
ax.set_xlabel('Actual Speed (mph)', fontsize=11, fontweight='bold')
ax.set_ylabel('Absolute Error (mph)', fontsize=11, fontweight='bold')
ax.set_title('Error vs Actual Speed', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Prediction analysis visualized")

## Part 7: Key Learnings & Concepts Summary

### 7.1 Graph Convolutional Networks: Key Insights

#### What Makes GCNs Special?
1. **Structured Data**: Operates on graphs, not just vectors/images
2. **Neighborhood Aggregation**: Each node learns from its neighbors
3. **Spatial Relationships**: Captures how traffic at one sensor influences others

#### The Mathematical Foundation
```
Forward Pass: H^(l+1) = D^(-1/2) A D^(-1/2) H^(l) W^(l) + b^(l)

Components:
  • A: Adjacency matrix (captures graph structure)
  • D: Degree matrix (normalization factor)
  • H: Hidden states (learned representations)
  • W: Learnable weights
  • b: Bias term
```

#### Why Normalize?
- **D^(-1/2) A D^(-1/2)**: Symmetric normalization prevents gradient issues
- Scales contributions by node degree
- Enables stable training on arbitrary graphs

### 7.2 Architecture of Our Model

```
Input: 12 temporal features per sensor
  ↓
GCN Layer 1: 12 → 32 (learn spatial patterns)
  ↓
ReLU Activation: Introduce non-linearity
  ↓
Dropout: Regularization (0.3 rate)
  ↓
GCN Layer 2: 32 → 1 (produce speed prediction)
  ↓  ┌──────────────────────────┐
  └─→ + Residual (12 → 1)    │ Skip connection
     └──────────────────────────┘
Output: 1 value (predicted speed)
```

**Why Two Paths?**
- **Graph Path (GCN)**: Learns spatial relationships from neighbors
- **Direct Path (Residual)**: Preserves temporal information
- **Combined**: Better predictions using both sources

### 7.3 Training Strategy

#### K-Fold Cross-Validation
- **5 folds**: Data split into 5 parts
- Each fold: 4 parts train, 1 part test
- Benefits: More robust evaluation, better use of limited data

#### Loss Function: MSE
```
Loss = (1/n) * Σ(y_true - y_pred)²
```
- Penalizes large errors more (quadratic)
- Standard for regression tasks

#### Optimization: Adam
- Adaptive learning rate per parameter
- Combines momentum and RMSprop
- Great for neural networks (converges quickly)

### 7.4 Evaluation Metrics

1. **MSE (Mean Squared Error)**
   - Measures average squared differences
   - Lower is better

2. **MAE (Mean Absolute Error)**
   - Average absolute error in mph
   - More interpretable than MSE

3. **R² Score (Coefficient of Determination)**
   - Proportion of variance explained
   - Range: (-∞, 1], where 1 = perfect
   - Formula: R² = 1 - (SS_res / SS_tot)

### 7.5 Common Challenges & Solutions

| Challenge | Root Cause | Solution |
|-----------|-----------|----------|
| Gradient Explosion | Large weight updates | Gradient clipping |
| Overfitting | Model too complex | Dropout, L2 regularization |
| Poor Convergence | High learning rate | Reduce learning rate, use optimizer |
| Sparse Graphs | Few connections | K-NN to ensure connectivity |
| Numerical Instability | Improper normalization | Symmetric normalization D^(-1/2)AD^(-1/2) |

### 7.6 Practical Tips for GCN Development

1. **Graph Construction**
   - K-NN ensures connectivity
   - Inverse distance weighting captures proximity
   - Symmetrization handles bidirectional relationships

2. **Feature Engineering**
   - Normalize features to similar scales
   - Use domain knowledge (temporal windows)
   - Consider multiple feature sources

3. **Model Design**
   - Start simple (2-3 layers)
   - Use residual connections
   - Balance model size with data

4. **Training**
   - Use cross-validation
   - Monitor validation metrics
   - Save best model, early stopping
   - Start with small learning rate

5. **Evaluation**
   - Multiple metrics (MSE, MAE, R²)
   - Analyze error distribution
   - Compare against baselines

## Part 8: Exercises & Next Steps

### Exercise 1: Modify the Network Architecture

Try adding more layers or changing hidden dimensions:
```python
class GCNDeep(nn.Module):
    def __init__(self, nfeat, nhid1, nhid2, nclass, dropout):
        super(GCNDeep, self).__init__()
        self.gc1 = GraphConvolution(nfeat, nhid1)    # First layer
        self.gc2 = GraphConvolution(nhid1, nhid2)    # NEW: Second hidden layer
        self.gc3 = GraphConvolution(nhid2, nclass)   # Output layer
        self.residual = nn.Linear(nfeat, nclass)
        # ... rest of implementation
```

**Questions:**
- How does deeper architecture affect performance?
- Does it converge faster or slower?
- What's the trade-off with computational cost?

### Exercise 2: Experiment with Hyperparameters

Create a hyperparameter sweep:
```python
learning_rates = [0.0001, 0.001, 0.01]
dropouts = [0.1, 0.3, 0.5]
hidden_sizes = [16, 32, 64]

results = {}
for lr in learning_rates:
    for dropout in dropouts:
        for hidden in hidden_sizes:
            # Train and evaluate
            results[(lr, dropout, hidden)] = accuracy
```

**Questions:**
- Which combination works best?
- How sensitive is the model to each parameter?
- Can you find patterns in what works well?

### Exercise 3: Analyze Attention Mechanisms

Extend the model with Graph Attention Networks (GATs):
```python
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, heads=4):
        # Use multi-head attention instead of fixed adjacency
        # Allows model to learn what neighbors matter most
```

**Questions:**
- How do attention weights compare to fixed adjacency?
- Do certain sensors matter more for predictions?
- Can you interpret what the model learned about traffic?

### Exercise 4: Real Dataset Implementation

When you have actual METR-LA data:
1. Load real traffic speeds instead of synthetic data
2. Use actual sensor locations and road network
3. Implement temporal sequences (predict multiple steps ahead)
4. Compare with baseline models (ARIMA, MLP)

**Expected Improvements:**
- Better predictions due to real spatial-temporal patterns
- Interpretable results (identify congestion hotspots)
- Production-ready performance

### Resources for Deeper Learning

**Papers:**
- [Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907) - Kipf & Welling (2017)
- [Graph Attention Networks](https://arxiv.org/abs/1710.10903) - Veličković et al. (2018)
- [Spatial-Temporal Graph Convolutional Networks](https://arxiv.org/abs/1909.06662) - Yu et al. (2019)

**Libraries:**
- **PyG (PyTorch Geometric)**: Graph neural network toolkit
- **DGL (Deep Graph Library)**: Graph deep learning framework
- **Spektral**: Graph neural networks in Keras/TensorFlow

**Datasets:**
- METR-LA Traffic Dataset
- PeMS-Bay
- NYC Taxi
- Social Networks (citation networks, collaboration graphs)

In [ ]:
print("\n" + "="*70)
print("CONGRATULATIONS!")
print("="*70)
print("\nYou've successfully learned:")
print("  ✓ Graph Convolutional Networks (GCN)")
print("  ✓ Graph normalization and preprocessing")
print("  ✓ Spatial-temporal modeling")
print("  ✓ Training with K-fold cross-validation")
print("  ✓ Inference and prediction")
print("  ✓ Evaluation metrics (MSE, MAE, R²)")
print("\nNext Steps:")
print("  1. Experiment with exercises above")
print("  2. Read the referenced papers")
print("  3. Try on real datasets (METR-LA, PeMS-Bay)")
print("  4. Explore advanced architectures (GAT, STGCN)")
print("  5. Deploy for production predictions")
print("="*70 + "\n")